In [6]:
# Cell 1: Import libraries and mount Google Drive
from google.colab import drive
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import euclidean_distances
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Path to the main folder containing shape folders
main_folder_path = '/content/drive/MyDrive/shapes'
output_folder_path = os.path.join(main_folder_path, "tsne_plots")  # Folder to save t-SNE plots

# Create the output folder if it doesn't exist
os.makedirs(output_folder_path, exist_ok=True)

Mounted at /content/drive


In [7]:
# Cell 2: Function to load images from a folder
def load_images_from_folder(folder_path):
    """
    Load all images from a given folder as flattened grayscale arrays.
    Separates original image from child drawings.
    Keeps original 200x200 resolution.
    """
    data = []
    labels = []
    original_data = None
    original_label = None

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".png"):  # Process only PNG files
            file_path = os.path.join(folder_path, file_name)
            image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
            if image is not None:
                # Verify the image is 200x200, if not, skip it
                if image.shape == (200, 200):
                    flattened_image = image.flatten()  # Flatten the 200x200 image (40,000 features)
                    label = file_name.split(".")[0]  # Use filename without extension as label

                    # Check if this is the original image
                    if file_name.startswith("original"):
                        original_data = flattened_image
                        original_label = label
                    else:
                        data.append(flattened_image)
                        labels.append(label)
                else:
                    print(f"Warning: {file_name} is not 200x200 (shape: {image.shape}), skipping...")

    return np.array(data), labels, original_data, original_label

In [8]:
# Cell 3: Function to apply t-SNE
def apply_tsne(data, n_components=2, perplexity=30, random_state=42):
    """
    Apply t-SNE for dimensionality reduction.
    """
    tsne = TSNE(n_components=n_components, perplexity=perplexity,
                random_state=random_state, n_iter=1000, verbose=1)
    reduced_data = tsne.fit_transform(data)
    return reduced_data

In [9]:
# Cell 4: Functions to create visualizations and calculate distances
def calculate_distances(reduced_data, original_reduced, labels):
    """
    Calculate Euclidean distances from original image to all child drawings in t-SNE space.
    """
    distances = []
    for i, point in enumerate(reduced_data):
        distance = euclidean_distances([original_reduced], [point])[0][0]
        distances.append({
            'child_id': labels[i],
            'distance': distance,
            'tsne_x': point[0],
            'tsne_y': point[1]
        })
    return distances

def save_tsne_plot_with_distances(reduced_data, labels, original_reduced, original_label,
                                output_file_path, shape_name, distances_df):
    """
    Save a t-SNE plot with the original image highlighted and distance information.
    """
    plt.figure(figsize=(14, 10))

    # Plot child drawings
    scatter = plt.scatter(reduced_data[:, 0], reduced_data[:, 1],
                         alpha=0.7, c=distances_df['distance'],
                         cmap='viridis_r', s=60)

    # Plot original image in bold/larger
    plt.scatter(original_reduced[0], original_reduced[1],
               color='red', s=200, marker='*',
               edgecolors='black', linewidth=2,
               label=f'Original ({original_label})')

    # Add labels for child drawings
    for i, label in enumerate(labels):
        plt.annotate(label, (reduced_data[i, 0], reduced_data[i, 1]),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8, alpha=0.8)

    # Add colorbar for distances
    cbar = plt.colorbar(scatter)
    cbar.set_label('Distance from Original', rotation=270, labelpad=15)

    plt.title(f"t-SNE Analysis for {shape_name} - Distances from Original")
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_file_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved t-SNE plot to {output_file_path}")

def load_and_display_images(folder_path, child_ids, title, output_file_path, distances_df):
    """
    Load and display images in a montage with scores and child IDs.
    """
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(title, fontsize=16, fontweight='bold')

    for i, child_id in enumerate(child_ids):
        row = i // 5
        col = i % 5

        # Load the image
        image_path = os.path.join(folder_path, f"{child_id}.png")
        if os.path.exists(image_path):
            image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            axes[row, col].imshow(image, cmap='gray')
        else:
            # If image not found, show empty plot
            axes[row, col].text(0.5, 0.5, 'Image\nNot Found',
                              ha='center', va='center', transform=axes[row, col].transAxes)

        # Get the distance score for this child
        distance = distances_df[distances_df['child_id'] == child_id]['distance'].iloc[0]

        # Set title with child ID and score
        axes[row, col].set_title(f'ID: {child_id}\nScore: {distance:.3f}',
                                fontsize=12, fontweight='bold')
        axes[row, col].axis('off')

    plt.tight_layout()
    plt.savefig(output_file_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved image montage to {output_file_path}")

def save_top_bottom_images_montage(distances_df, folder_path, output_file_path, shape_name):
    """
    Save image montages showing top 5 and bottom 5 drawings with their scores.
    """
    # Get top 5 (closest) and bottom 5 (farthest)
    top_5 = distances_df.nsmallest(5, 'distance')
    bottom_5 = distances_df.nlargest(5, 'distance')

    # Combine top 5 and bottom 5
    combined_ids = list(top_5['child_id']) + list(bottom_5['child_id'])

    # Create the montage
    title = f'{shape_name} - Top 5 Closest (Row 1) and Top 5 Farthest (Row 2)'
    load_and_display_images(folder_path, combined_ids, title, output_file_path, distances_df)

def save_original_image_display(folder_path, original_label, output_file_path, shape_name):
    """
    Save a display of the original image for reference.
    """
    original_path = os.path.join(folder_path, f"{original_label}.png")

    if os.path.exists(original_path):
        original_image = cv2.imread(original_path, cv2.IMREAD_GRAYSCALE)

        plt.figure(figsize=(8, 8))
        plt.imshow(original_image, cmap='gray')
        plt.title(f'Original Reference Image - {shape_name}', fontsize=16, fontweight='bold')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(output_file_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved original image display to {output_file_path}")
    else:
        print(f"Original image not found at {original_path}")

In [10]:
# Cell 5: Function to process all folders with distance analysis
def process_folders_for_tsne(main_folder_path, output_folder_path):
    """
    Process each shape folder, apply t-SNE, calculate distances, and save the plots and scores.
    """
    all_results = {}

    for shape_folder in os.listdir(main_folder_path):
        shape_folder_path = os.path.join(main_folder_path, shape_folder)

        # Check if it's a valid shape folder
        if os.path.isdir(shape_folder_path) and shape_folder.startswith("shape"):
            print(f"\nProcessing {shape_folder}...")

            # Load images and separate original from child drawings
            data, labels, original_data, original_label = load_images_from_folder(shape_folder_path)

            if data.shape[0] == 0:
                print(f"No child drawing images found in {shape_folder}. Skipping.")
                continue

            if original_data is None:
                print(f"No original image found in {shape_folder}. Skipping.")
                continue

            print(f"Loaded {data.shape[0]} child drawings and 1 original image for {shape_folder}.")

            # Combine all data for t-SNE (original + child drawings)
            all_data = np.vstack([original_data.reshape(1, -1), data])

            # Apply t-SNE to all images
            print(f"Applying t-SNE to {shape_folder}...")
            reduced_data_all = apply_tsne(all_data)

            # Separate original and child drawings in reduced space
            original_reduced = reduced_data_all[0]
            child_reduced = reduced_data_all[1:]

            # Calculate distances from original to each child drawing
            distances_info = calculate_distances(child_reduced, original_reduced, labels)
            distances_df = pd.DataFrame(distances_info)
            distances_df = distances_df.sort_values('distance')

            # Save results
            all_results[shape_folder] = distances_df

            # 1. Save t-SNE plot with distances visualization
            tsne_plot_path = os.path.join(output_folder_path, f"{shape_folder}_tsne_distances.png")
            save_tsne_plot_with_distances(child_reduced, labels, original_reduced,
                                       original_label, tsne_plot_path, shape_folder, distances_df)

            # 2. Save original image for reference
            original_display_path = os.path.join(output_folder_path, f"{shape_folder}_original_reference.png")
            save_original_image_display(shape_folder_path, original_label, original_display_path, shape_folder)

            # 3. Save top/bottom images montage
            images_montage_path = os.path.join(output_folder_path, f"{shape_folder}_top_bottom_images.png")
            save_top_bottom_images_montage(distances_df, shape_folder_path, images_montage_path, shape_folder)

            # 4. Save scores list to CSV
            scores_csv_path = os.path.join(output_folder_path, f"{shape_folder}_all_scores.csv")
            distances_df.to_csv(scores_csv_path, index=False)
            print(f"Saved all scores to {scores_csv_path}")

            # Display summary statistics
            print(f"Distance statistics for {shape_folder}:")
            print(f"  Mean distance: {distances_df['distance'].mean():.3f}")
            print(f"  Std deviation: {distances_df['distance'].std():.3f}")
            print(f"  Min distance: {distances_df['distance'].min():.3f} (Child ID: {distances_df.iloc[0]['child_id']})")
            print(f"  Max distance: {distances_df['distance'].max():.3f} (Child ID: {distances_df.iloc[-1]['child_id']})")

    return all_results

In [11]:
# Cell 6: Main function and execution
def main():
    # Process each shape folder and generate t-SNE plots with distance analysis
    results = process_folders_for_tsne(main_folder_path, output_folder_path)

    # Print summary of all processed shapes
    print(f"\n{'='*50}")
    print("SUMMARY OF ALL PROCESSED SHAPES (t-SNE)")
    print(f"{'='*50}")

    for shape_name, distances_df in results.items():
        print(f"\n{shape_name}:")
        print(f"  Total drawings: {len(distances_df)}")
        print(f"  Mean distance from original: {distances_df['distance'].mean():.3f}")
        print(f"  Best match (closest): {distances_df.iloc[0]['child_id']} (distance: {distances_df.iloc[0]['distance']:.3f})")
        print(f"  Worst match (farthest): {distances_df.iloc[-1]['child_id']} (distance: {distances_df.iloc[-1]['distance']:.3f})")

    return results

# Run the main function
results = main()


Processing shape5...
Loaded 893 child drawings and 1 original image for shape5.
Applying t-SNE to shape5...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 894 samples in 0.013s...
[t-SNE] Computed neighbors for 894 samples in 1.074s...
[t-SNE] Computed conditional probabilities for sample 894 / 894
[t-SNE] Mean sigma: 1691.264450
[t-SNE] KL divergence after 250 iterations with early exaggeration: 94.165375
[t-SNE] KL divergence after 1000 iterations: 1.999964
Saved t-SNE plot to /content/drive/MyDrive/shapes/tsne_plots/shape5_tsne_distances.png
Saved original image display to /content/drive/MyDrive/shapes/tsne_plots/shape5_original_reference.png
Saved image montage to /content/drive/MyDrive/shapes/tsne_plots/shape5_top_bottom_images.png
Saved all scores to /content/drive/MyDrive/shapes/tsne_plots/shape5_all_scores.csv
Distance statistics for shape5:
  Mean distance: 15.420
  Std deviation: 6.641
  Min distance: 0.285 (Child ID: 11058)
  Max distance: 29.653 (Child ID: 10401

In [13]:
# Cell 8: Additional analysis functions (optional - run after main analysis)

def display_shape_results(results, shape_name):
    """
    Display detailed results for a specific shape.
    """
    if shape_name in results:
        df = results[shape_name]
        print(f"\nDetailed t-SNE results for {shape_name}:")
        print(f"{'Child ID':<10} {'Distance':<12} {'t-SNE X':<12} {'t-SNE Y':<12}")
        print("-" * 50)
        for _, row in df.iterrows():
            print(f"{row['child_id']:<10} {row['distance']:<12.3f} {row['tsne_x']:<12.3f} {row['tsne_y']:<12.3f}")
    else:
        print(f"Shape {shape_name} not found in results.")

def compare_shapes_performance(results):
    """
    Compare performance across all shapes using t-SNE.
    """
    comparison_data = []
    for shape_name, df in results.items():
        comparison_data.append({
            'shape': shape_name,
            'num_drawings': len(df),
            'mean_distance': df['distance'].mean(),
            'std_distance': df['distance'].std(),
            'min_distance': df['distance'].min(),
            'max_distance': df['distance'].max(),
            'best_child': df.iloc[0]['child_id'],
            'worst_child': df.iloc[-1]['child_id']
        })

    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.sort_values('mean_distance')

    print("\nShape Performance Comparison using t-SNE (sorted by mean distance):")
    print(comparison_df.to_string(index=False))

    return comparison_df


display_shape_results(results, 'shape3')
comparison_df = compare_shapes_performance(results)  # Compare all shapes


Detailed t-SNE results for shape3:
Child ID   Distance     t-SNE X      t-SNE Y     
--------------------------------------------------
10919      1.844        -6.279       41.064      
10300      3.747        -1.382       43.034      
10511      3.762        -1.972       44.580      
10391      4.624        -0.466       42.265      
10367      5.446        -3.686       37.207      
10360      5.594        -10.569      43.578      
12502      5.758        -5.463       48.215      
10543      6.004        -1.974       37.335      
11204      6.503        -11.349      44.223      
7577       6.705        -8.749       36.854      
10308      6.885        -0.842       37.048      
7747       7.652        -12.738      42.507      
10613      7.751        -11.702      46.507      
11073      8.375        -6.205       34.170      
10750      8.455        0.575        36.190      
10952      8.966        2.398        47.408      
13013      9.172        -6.170       51.577      
10443      9.